# STEP07 — Moth: Focal Loss γ=2 continuation from EXP045 @50k

Цель — **чистый controlled experiment** против уже существующего EXP047-B risk-loss.

Замораживаем всё, кроме presence loss:

- assembly: `ilPloInte3.2`
- architecture: WIDTH512, theoretical RF=1029
- branch point: тот же EXP045 checkpoint @50k
- optimizer state / Adam moments: те же
- generator RNG / torch RNG: те же
- batch size: 4
- continuation: +20k
- LR: warm-up 500 шагов `3e-5 → 1e-4`, затем cosine `1e-4 → 1e-5`
- intensity objective: тот же Huber, weight=0.1
- validation: **только один полный проход после 20k**
- validation не используется для checkpoint selection

Единственное изменение:

`class-balanced BCE → class-balanced Focal Loss, gamma=2.0`

`alpha` отдельно не вводим: class balancing уже содержится в существующих `plus_loss_weight /
minus_loss_weight_reverse`; дополнительный alpha смешал бы два механизма балансировки.

Это exploratory experiment: этот validation split уже многократно смотрели раньше.


In [1]:
from pathlib import Path
import copy
import gc
import hashlib
import json
import math
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch_directml

EXPECTED_PYTHON = Path(
    r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe"
).resolve()
assert Path(sys.executable).resolve() == EXPECTED_PYTHON, (
    sys.executable,
    EXPECTED_PYTHON,
)

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (current, *current.parents)
    if (p / "README.md").is_file()
    and (p / "src/ml_project").is_dir()
    and (p / "data/raw/unpacked").is_dir()
)

AGENTS = next(
    (
        p for p in (
            PROJECT_ROOT / "AGENTS.md",
            PROJECT_ROOT.parent / "AGENTS.md",
        )
        if p.is_file()
    ),
    None,
)
assert AGENTS is not None

EXPECTED_AGENTS_SHA = "bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3"
agents_sha = hashlib.sha256(AGENTS.read_bytes()).hexdigest()
assert agents_sha == EXPECTED_AGENTS_SHA, (agents_sha, EXPECTED_AGENTS_SHA)

LOADER = PROJECT_ROOT / "src/ml_project/epic_data.py"
EXPECTED_LOADER_SHA = "37c9b0ed4ccecc02b484010c7b5b02fa35fa35f4c976e76e6d0e00096380b468"
loader_sha = hashlib.sha256(LOADER.read_bytes()).hexdigest()
assert loader_sha == EXPECTED_LOADER_SHA, (loader_sha, EXPECTED_LOADER_SHA)

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    evaluate_presence_intensity_model,
    forward_both_strands_multitask,
)
from ml_project.epic_data import SplitConfig, load_split

print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)
print("GPU:", torch_directml.device_name(0))


Project: D:\Projects\EPIC\EPIC
Python: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe
GPU: AMD Radeon RX 6750 GRE 12GB 


In [2]:
# Frozen experiment configuration.
ASSEMBLY = "ilPloInte3.2"
EXPERIMENT_ID = "SUBMIT-MOTH-FOCAL-GAMMA2"
EXPERIMENT_SLUG = "SUBMIT_ilPloInte3p2_FOCAL_GAMMA2_CONTINUATION_20K"

BASE_STEP = 50_000
CONTINUATION_STEPS = 20_000
BATCH_SIZE = 4
LOG_EVERY = 250
LATEST_EVERY = 500
SNAPSHOT_STEPS = {2_000, 5_000, 10_000, 15_000, 20_000}

WARMUP_STEPS = 500
START_LR = 3e-5
PEAK_LR = 1e-4
END_LR = 1e-5

INTENSITY_LOSS_WEIGHT = 0.1
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = None

EXPECTED_CHANNELS = 512
EXPECTED_RF = 1029
EXPECTED_PARAMETERS = 14_708_738

BASE_RUN = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "SUBMIT_ilPloInte3p2_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029"
    / "run_001"
)
BASE_CHECKPOINT = BASE_RUN / "model_step_50000.pt"
EXPECTED_BASE_SHA = "01dd1761937dbe15ce334689db27221cc0801514a2bf9a7e63ee81ec9d521048"

RISK_RUN = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "SUBMIT_ilPloInte3p2_EXP047B_RISK_CONTINUATION"
    / "run_001"
)
RISK_CHECKPOINT = RISK_RUN / "branch_B" / "branch_B_final.pt"
EXPECTED_RISK_SHA = "975adb5184bff1ef5c760db94d69ac016e9b55084a3a0dfefab2a8e9d32cfed5"

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / EXPERIMENT_SLUG
    / "run_001"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

LATEST = RUN_DIR / "latest.pt"
FINAL = RUN_DIR / "focal_gamma2_final.pt"
HISTORY_PATH = RUN_DIR / "training_history.csv"
METRICS_PATH = RUN_DIR / "validation_metrics_report.csv"
STRONG_METRICS_PATH = RUN_DIR / "strong_validation_metrics.csv"
RESULT_PATH = RUN_DIR / "result_summary.json"
PREREG_PATH = RUN_DIR / "preregistration.json"

def sha256_file(path, block=8 * 1024 * 1024):
    d = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            d.update(chunk)
    return d.hexdigest()

assert BASE_CHECKPOINT.is_file(), BASE_CHECKPOINT
assert sha256_file(BASE_CHECKPOINT) == EXPECTED_BASE_SHA
if RISK_CHECKPOINT.is_file():
    assert sha256_file(RISK_CHECKPOINT) == EXPECTED_RISK_SHA

def planned_lr(local_step):
    assert 1 <= local_step <= CONTINUATION_STEPS
    if local_step <= WARMUP_STEPS:
        fraction = local_step / WARMUP_STEPS
        return START_LR + fraction * (PEAK_LR - START_LR)
    fraction = (
        (local_step - WARMUP_STEPS)
        / (CONTINUATION_STEPS - WARMUP_STEPS)
    )
    return END_LR + 0.5 * (PEAK_LR - END_LR) * (
        1.0 + math.cos(math.pi * fraction)
    )

prereg = {
    "experiment_id": EXPERIMENT_ID,
    "assembly": ASSEMBLY,
    "branch_point_step": BASE_STEP,
    "base_checkpoint": str(BASE_CHECKPOINT),
    "base_checkpoint_sha256": EXPECTED_BASE_SHA,
    "only_changed_component": "presence objective",
    "presence_objective": "class-balanced focal loss",
    "focal_gamma": FOCAL_GAMMA,
    "focal_alpha": FOCAL_ALPHA,
    "reason_alpha_none": (
        "existing class-balanced loss weights are preserved; "
        "no second class-balancing coefficient"
    ),
    "continuation_steps": CONTINUATION_STEPS,
    "batch_size": BATCH_SIZE,
    "warmup_steps": WARMUP_STEPS,
    "start_lr": START_LR,
    "peak_lr": PEAK_LR,
    "end_lr": END_LR,
    "intensity_loss_weight": INTENSITY_LOSS_WEIGHT,
    "validation_policy": (
        "one full natural validation scan after 20k; "
        "no validation checkpoint selection"
    ),
    "comparison_targets": {
        "EXP045_AP": 0.158191438225949,
        "EXP047B_AP": 0.1700035818376157,
        "EXP045_Spearman": 0.15125935941947766,
        "EXP047B_Spearman": 0.1698961163371729,
    },
    "caveat": (
        "Exploratory validation: this validation split has been "
        "inspected in prior experiments."
    ),
}

if PREREG_PATH.exists():
    old = json.loads(PREREG_PATH.read_text(encoding="utf-8"))
    assert old == prereg, "Refusing changed preregistration"
else:
    PREREG_PATH.write_text(
        json.dumps(prereg, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

print("Base:", BASE_CHECKPOINT)
print("Run:", RUN_DIR)
print(
    "LR probes:",
    planned_lr(1),
    planned_lr(WARMUP_STEPS),
    planned_lr(CONTINUATION_STEPS),
)
print("Pre-registration: SEALED")


Base: D:\Projects\EPIC\EPIC\artifacts\experiments\SUBMIT_ilPloInte3p2_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029\run_001\model_step_50000.pt
Run: D:\Projects\EPIC\EPIC\artifacts\experiments\SUBMIT_ilPloInte3p2_FOCAL_GAMMA2_CONTINUATION_20K\run_001
LR probes: 3.014e-05 0.0001 1e-05
Pre-registration: SEALED


In [3]:
# Deterministic moth split + geometry.
SPLIT_CONFIG = SplitConfig(
    assembly=ASSEMBLY,
    split_version="contig_holdout_v1",
    seed=42,
    validation_contig_fraction=0.25,
)
split = load_split(PROJECT_ROOT, SPLIT_CONFIG, verify_hashes=True)
display(split.summary)

train_row = split.summary.loc[split.summary["split"]=="train"].iloc[0]
val_row = split.summary.loc[split.summary["split"]=="validation"].iloc[0]
test_row = split.summary.loc[split.summary["split"]=="test"].iloc[0]

assert int(train_row["contigs"]) == 19
assert int(val_row["contigs"]) == 7
assert int(test_row["contigs"]) == 7
assert int(train_row["positive_positions"]) == 581_052
assert int(val_row["positive_positions"]) == 162_306
assert int(val_row["position_strand_count"]) == 104_114_200
assert split.positive_targets("test") is None

EXPECTED_VAL_POSITIONS = int(val_row["position_strand_count"])
EXPECTED_VAL_POSITIVES = int(val_row["positive_positions"])

sequences, fasta_alphabet = load_baseline_sequences(split)

base = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=False)
assert int(base["step"]) == BASE_STEP
assert base["assembly"] == ASSEMBLY

profile_config = ProfileWindowConfig(**base["profile_config"])
assert profile_config.target_length == 1024
assert profile_config.context_flank == 130
assert profile_config.positive_branch_fraction == 0.5

generator_probe = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
probe = generator_probe.sample_batch(BATCH_SIZE)
assert tuple(probe.x_both_strands.shape) == (8, 5, 1284)

model_probe = create_cnn_presence_intensity(channels=EXPECTED_CHANNELS)
assert sum(p.numel() for p in model_probe.parameters()) == EXPECTED_PARAMETERS

print("MOTH FOCAL READINESS: PASS")

del generator_probe, probe, model_probe
gc.collect()


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,19,178622,164558039,329116078,60.145242,581052.0,328535026.0,0.176549
1,validation,7,63150,52057100,104114200,19.026642,162306.0,103951894.0,0.155892
2,test,7,67434,56985954,113971908,20.828116,NaN,NaN,NaN


MOTH FOCAL READINESS: PASS


40

## Focal loss

Используем стандартную бинарную focal-модуляцию:

\[
FL = (1-p_t)^\gamma \cdot BCE
\]

где `γ=2`.

Существующие class-balanced веса применяются **после** focal-модуляции.
Нормализация остаётся как в текущем weighted profile BCE: отдельный weighted mean
для `+` и `-`, затем сумма двух strand losses.

Перед обучением есть regression check: при `gamma=0` новая функция обязана совпасть
с существующей `dml_weighted_profile_bce_with_logits`. Если не совпадает — long run не стартует.


In [4]:
def _stable_bce_elementwise(logits, targets):
    # Stable BCEWithLogits:
    # max(z,0) - z*y + log(1 + exp(-|z|))
    return (
        torch.clamp(logits, min=0.0)
        - logits * targets
        + torch.log1p(torch.exp(-torch.abs(logits)))
    )

def _weighted_focal_one_strand(logits, targets, weights, gamma):
    bce = _stable_bce_elementwise(logits, targets)
    p_t = torch.exp(-bce)
    focal_factor = torch.pow(1.0 - p_t, gamma)

    # IMPORTANT:
    # Existing dml_weighted_profile_bce_with_logits uses a weighted SUM,
    # not a weighted mean. Preserve that exact scale so gamma=0 is
    # algebraically identical to the current BCE objective.
    return torch.sum(weights * focal_factor * bce)

def dml_weighted_profile_focal_with_logits(
    plus_logits,
    minus_logits,
    plus_targets,
    minus_targets,
    plus_weights,
    minus_weights,
    gamma=2.0,
):
    return (
        _weighted_focal_one_strand(
            plus_logits, plus_targets, plus_weights, gamma
        )
        + _weighted_focal_one_strand(
            minus_logits, minus_targets, minus_weights, gamma
        )
    )

# CPU algebra/regression test: gamma=0 must reduce exactly to weighted BCE.
torch.manual_seed(1234)
z1 = torch.randn(2, 1, 17)
z2 = torch.randn(2, 1, 17)
y1 = torch.randint(0, 2, z1.shape).float()
y2 = torch.randint(0, 2, z2.shape).float()
w1 = 0.1 + torch.rand(z1.shape)
w2 = 0.1 + torch.rand(z2.shape)

reference = dml_weighted_profile_bce_with_logits(
    z1, z2, y1, y2, w1, w2
)
focal0 = dml_weighted_profile_focal_with_logits(
    z1, z2, y1, y2, w1, w2, gamma=0.0
)

print("reference BCE:", float(reference))
print("focal gamma=0:", float(focal0))
print("absolute delta:", abs(float(reference) - float(focal0)))

assert torch.allclose(reference, focal0, rtol=1e-6, atol=1e-7), (
    float(reference),
    float(focal0),
)

print("FOCAL γ=0 == EXISTING BCE: PASS")


reference BCE: 31.316253662109375
focal gamma=0: 31.316253662109375
absolute delta: 0.0
FOCAL γ=0 == EXISTING BCE: PASS


## DirectML one-step smoke

Это **не** часть обучения. Загружается EXP045 @50k, делается один шаг focal loss,
проверяется backward/optimizer/DirectML fallback, затем всё удаляется.

Long run после этого заново загружает исходный @50k checkpoint и восстанавливает RNG,
поэтому smoke не меняет последовательность training batches.


In [5]:
device = torch_directml.device()

smoke_base = torch.load(
    BASE_CHECKPOINT, map_location="cpu", weights_only=False
)
smoke_gen = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
smoke_gen.rng.bit_generator.state = copy.deepcopy(
    smoke_base["generator_rng_state"]
)
torch.set_rng_state(smoke_base["torch_rng_state"])

smoke_model = create_cnn_presence_intensity(
    channels=EXPECTED_CHANNELS
)
smoke_model.load_state_dict(smoke_base["model_state_dict"])
smoke_model = smoke_model.to(device)

smoke_opt = DirectMLAdam(
    smoke_model.parameters(),
    lr=planned_lr(1),
)
smoke_opt.load_state_dict(smoke_base["optimizer_state_dict"])
for group in smoke_opt.param_groups:
    group["lr"] = planned_lr(1)

smoke_model.train()
batch = smoke_gen.sample_batch(BATCH_SIZE)

pc = batch.count[:, 0:1, :].contiguous()
mc = batch.count[:, 1:2, :].flip(-1).contiguous()

smoke_opt.zero_grad(set_to_none=True)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    pp, mp, pi, mi = forward_both_strands_multitask(
        smoke_model,
        batch.x_both_strands.to(device),
        batch_size=BATCH_SIZE,
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

    presence_loss = dml_weighted_profile_focal_with_logits(
        pp,
        mp,
        batch.plus_target.to(device),
        batch.minus_target_reverse.to(device),
        batch.plus_loss_weight.to(device),
        batch.minus_loss_weight_reverse.to(device),
        gamma=FOCAL_GAMMA,
    )

    intensity_loss = (
        dml_weighted_huber(
            pi,
            torch.log1p(pc).to(device),
            (
                2.0
                * batch.plus_loss_weight
                * (pc > 0).float()
            ).to(device),
            delta=1.0,
        )
        + dml_weighted_huber(
            mi,
            torch.log1p(mc).to(device),
            (
                2.0
                * batch.minus_loss_weight_reverse
                * (mc > 0).float()
            ).to(device),
            delta=1.0,
        )
    )

    total_loss = (
        presence_loss
        + INTENSITY_LOSS_WEIGHT * intensity_loss
    )

    vals = [
        float(x.detach().cpu().item())
        for x in (presence_loss, intensity_loss, total_loss)
    ]
    assert all(math.isfinite(v) for v in vals), vals

    total_loss.backward()
    smoke_opt.step()

fallback = [
    str(w.message)
    for w in caught
    if "fall back" in str(w.message).lower()
]
assert not fallback, fallback

print(
    "SMOKE PASS | "
    f"focal={vals[0]:.6f} "
    f"intensity={vals[1]:.6f} "
    f"total={vals[2]:.6f}"
)

del (
    smoke_base, smoke_gen, smoke_model, smoke_opt,
    batch, pc, mc, pp, mp, pi, mi,
    presence_loss, intensity_loss, total_loss,
)
gc.collect()


SMOKE PASS | focal=0.045783 intensity=0.188387 total=0.064621


97

## Long training: +20k Focal γ=2

Можно перезапускать ячейку: `latest.pt` сохраняется каждые 500 шагов.
Snapshot'ы сохраняются на 2k/5k/10k/15k/20k, но **не оцениваются на validation**
до завершения 20k.


In [6]:
def model_state_on_cpu(model):
    return {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }

def tree_to_cpu(x):
    if torch.is_tensor(x):
        return x.detach().cpu().clone()
    if isinstance(x, dict):
        return {k: tree_to_cpu(v) for k, v in x.items()}
    if isinstance(x, list):
        return [tree_to_cpu(v) for v in x]
    if isinstance(x, tuple):
        return tuple(tree_to_cpu(v) for v in x)
    return x

def atomic_torch_save(path, payload):
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)

def save_payload(
    path,
    local_step,
    model,
    optimizer,
    generator,
    history,
    seen,
    elapsed,
):
    payload = {
        "experiment": EXPERIMENT_ID,
        "assembly": ASSEMBLY,
        "branch": "FOCAL_GAMMA2",
        "base_step": BASE_STEP,
        "local_step": int(local_step),
        "step": BASE_STEP + int(local_step),
        "base_checkpoint_sha256": EXPECTED_BASE_SHA,
        "model_state_dict": model_state_on_cpu(model),
        "optimizer_state_dict": tree_to_cpu(
            optimizer.state_dict()
        ),
        "generator_rng_state": copy.deepcopy(
            generator.rng.bit_generator.state
        ),
        "torch_rng_state": torch.get_rng_state(),
        "model_config": {
            "input_channels": 5,
            "channels": EXPECTED_CHANNELS,
            "output_channels": 2,
            "theoretical_rf": EXPECTED_RF,
        },
        "profile_config": {
            "target_length": profile_config.target_length,
            "context_flank": profile_config.context_flank,
            "positive_branch_fraction":
                profile_config.positive_branch_fraction,
        },
        "training_config": {
            "continuation_steps": CONTINUATION_STEPS,
            "batch_size": BATCH_SIZE,
            "warmup_steps": WARMUP_STEPS,
            "start_lr": START_LR,
            "peak_lr": PEAK_LR,
            "end_lr": END_LR,
            "presence_objective": "class-balanced focal",
            "focal_gamma": FOCAL_GAMMA,
            "focal_alpha": FOCAL_ALPHA,
            "intensity_loss_weight":
                INTENSITY_LOSS_WEIGHT,
        },
        "seen_position_strand": int(seen),
        "elapsed_seconds": float(elapsed),
        "history": list(history),
    }
    atomic_torch_save(path, payload)

if FINAL.is_file():
    done = torch.load(
        FINAL, map_location="cpu", weights_only=False
    )
    assert int(done["local_step"]) == CONTINUATION_STEPS
    assert done["base_checkpoint_sha256"] == EXPECTED_BASE_SHA
    assert done["training_config"]["focal_gamma"] == FOCAL_GAMMA
    print("Already complete:", FINAL)
    del done

else:
    base = torch.load(
        BASE_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    generator = ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="train",
        config=profile_config,
        seed=42,
    )
    model = create_cnn_presence_intensity(
        channels=EXPECTED_CHANNELS
    )

    if LATEST.is_file():
        resume = torch.load(
            LATEST, map_location="cpu", weights_only=False
        )
        assert resume["experiment"] == EXPERIMENT_ID
        assert resume["assembly"] == ASSEMBLY
        assert resume["base_checkpoint_sha256"] == EXPECTED_BASE_SHA
        assert (
            resume["training_config"]["focal_gamma"]
            == FOCAL_GAMMA
        )

        model.load_state_dict(resume["model_state_dict"])
        generator.rng.bit_generator.state = copy.deepcopy(
            resume["generator_rng_state"]
        )
        torch.set_rng_state(resume["torch_rng_state"])

        start_step = int(resume["local_step"])
        history = list(resume["history"])
        seen = int(resume["seen_position_strand"])
        previous_elapsed = float(resume["elapsed_seconds"])

        model = model.to(device)
        optimizer = DirectMLAdam(
            model.parameters(),
            lr=planned_lr(max(start_step, 1)),
        )
        optimizer.load_state_dict(
            resume["optimizer_state_dict"]
        )

        print("RESUME:", LATEST, "at", start_step)
        del resume

    else:
        model.load_state_dict(base["model_state_dict"])
        generator.rng.bit_generator.state = copy.deepcopy(
            base["generator_rng_state"]
        )
        torch.set_rng_state(base["torch_rng_state"])

        start_step = 0
        history = []
        seen = 0
        previous_elapsed = 0.0

        model = model.to(device)
        optimizer = DirectMLAdam(
            model.parameters(),
            lr=planned_lr(1),
        )
        optimizer.load_state_dict(
            base["optimizer_state_dict"]
        )

    del base
    gc.collect()

    model.train()

    ema_presence = (
        history[-1]["ema_presence"]
        if history else None
    )
    ema_intensity = (
        history[-1]["ema_intensity"]
        if history else None
    )
    ema_total = (
        history[-1]["ema_total"]
        if history else None
    )

    started = time.perf_counter()

    for local_step in range(
        start_step + 1,
        CONTINUATION_STEPS + 1,
    ):
        lr = planned_lr(local_step)
        for group in optimizer.param_groups:
            group["lr"] = lr

        batch = generator.sample_batch(BATCH_SIZE)

        x_gpu = batch.x_both_strands.to(device)
        pt_gpu = batch.plus_target.to(device)
        mt_gpu = batch.minus_target_reverse.to(device)
        pw_gpu = batch.plus_loss_weight.to(device)
        mw_gpu = batch.minus_loss_weight_reverse.to(device)

        pc = batch.count[:, 0:1, :].contiguous()
        mc = batch.count[:, 1:2, :].flip(-1).contiguous()

        pit = torch.log1p(pc).to(device)
        mit = torch.log1p(mc).to(device)

        piw = (
            2.0
            * batch.plus_loss_weight
            * (pc > 0).float()
        ).to(device)
        miw = (
            2.0
            * batch.minus_loss_weight_reverse
            * (mc > 0).float()
        ).to(device)

        optimizer.zero_grad(set_to_none=True)

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            pp, mp, pi, mi = (
                forward_both_strands_multitask(
                    model,
                    x_gpu,
                    batch_size=BATCH_SIZE,
                    context_flank=profile_config.context_flank,
                    target_length=profile_config.target_length,
                )
            )

            presence_loss = (
                dml_weighted_profile_focal_with_logits(
                    pp,
                    mp,
                    pt_gpu,
                    mt_gpu,
                    pw_gpu,
                    mw_gpu,
                    gamma=FOCAL_GAMMA,
                )
            )

            intensity_loss = (
                dml_weighted_huber(
                    pi, pit, piw, delta=1.0
                )
                + dml_weighted_huber(
                    mi, mit, miw, delta=1.0
                )
            )

            total_loss = (
                presence_loss
                + INTENSITY_LOSS_WEIGHT
                * intensity_loss
            )

            vals = [
                float(v.detach().cpu().item())
                for v in (
                    presence_loss,
                    intensity_loss,
                    total_loss,
                )
            ]
            assert all(
                math.isfinite(v) for v in vals
            ), vals

            total_loss.backward()
            optimizer.step()

        fallback = [
            str(w.message)
            for w in caught
            if "fall back" in str(w.message).lower()
        ]
        assert not fallback, fallback

        presence_value, intensity_value, total_value = vals

        if ema_presence is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value
        else:
            ema_presence = (
                0.99 * ema_presence
                + 0.01 * presence_value
            )
            ema_intensity = (
                0.99 * ema_intensity
                + 0.01 * intensity_value
            )
            ema_total = (
                0.99 * ema_total
                + 0.01 * total_value
            )

        seen += int(batch.mask.sum())
        elapsed = (
            previous_elapsed
            + time.perf_counter()
            - started
        )

        if (
            local_step % LOG_EVERY == 0
            or local_step == CONTINUATION_STEPS
        ):
            run_elapsed = max(
                elapsed - previous_elapsed,
                1e-9,
            )
            completed_now = local_step - start_step
            sec_per_step = (
                run_elapsed / max(completed_now, 1)
            )

            record = {
                "local_step": local_step,
                "step": BASE_STEP + local_step,
                "learning_rate": lr,
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": float(ema_presence),
                "ema_intensity": float(ema_intensity),
                "ema_total": float(ema_total),
                "elapsed_seconds": float(elapsed),
                "seconds_per_step_current_process":
                    float(sec_per_step),
                "estimated_hours_remaining": float(
                    (
                        CONTINUATION_STEPS
                        - local_step
                    )
                    * sec_per_step
                    / 3600
                ),
                "seen_position_strand": int(seen),
            }
            history.append(record)
            pd.DataFrame(history).to_csv(
                HISTORY_PATH,
                index=False,
            )

            print(
                f"FOCAL local={local_step:05d}/"
                f"{CONTINUATION_STEPS} "
                f"global={BASE_STEP+local_step} "
                f"lr={lr:.7g} "
                f"loss=({presence_value:.5f},"
                f"{intensity_value:.5f},"
                f"{total_value:.5f}) "
                f"sec/step={sec_per_step:.3f} "
                f"ETA={record['estimated_hours_remaining']:.2f}h",
                flush=True,
            )

        if (
            local_step % LATEST_EVERY == 0
            or local_step == CONTINUATION_STEPS
        ):
            save_payload(
                LATEST,
                local_step,
                model,
                optimizer,
                generator,
                history,
                seen,
                elapsed,
            )

            if local_step in SNAPSHOT_STEPS:
                save_payload(
                    RUN_DIR
                    / f"model_local_step_{local_step:05d}.pt",
                    local_step,
                    model,
                    optimizer,
                    generator,
                    history,
                    seen,
                    elapsed,
                )

            if local_step == CONTINUATION_STEPS:
                save_payload(
                    FINAL,
                    local_step,
                    model,
                    optimizer,
                    generator,
                    history,
                    seen,
                    elapsed,
                )

        del (
            batch, x_gpu, pt_gpu, mt_gpu, pw_gpu, mw_gpu,
            pc, mc, pit, mit, piw, miw,
            pp, mp, pi, mi,
            presence_loss, intensity_loss, total_loss,
        )

    print("FOCAL TRAINING COMPLETE:", FINAL)

    del model, optimizer, generator
    gc.collect()


FOCAL local=00250/20000 global=50250 lr=6.5e-05 loss=(0.03122,0.37028,0.06825) sec/step=0.233 ETA=1.28h
FOCAL local=00500/20000 global=50500 lr=0.0001 loss=(0.08181,0.14179,0.09599) sec/step=0.231 ETA=1.25h
FOCAL local=00750/20000 global=50750 lr=9.99635e-05 loss=(0.04597,0.17030,0.06300) sec/step=0.231 ETA=1.23h
FOCAL local=01000/20000 global=51000 lr=9.985408e-05 loss=(0.09563,0.25550,0.12118) sec/step=0.230 ETA=1.21h
FOCAL local=01250/20000 global=51250 lr=9.96719e-05 loss=(0.04356,0.17979,0.06154) sec/step=0.230 ETA=1.20h
FOCAL local=01500/20000 global=51500 lr=9.941726e-05 loss=(0.12611,0.19479,0.14559) sec/step=0.230 ETA=1.18h
FOCAL local=01750/20000 global=51750 lr=9.909058e-05 loss=(0.06518,0.08738,0.07392) sec/step=0.230 ETA=1.17h
FOCAL local=02000/20000 global=52000 lr=9.869238e-05 loss=(0.07781,0.04426,0.08224) sec/step=0.230 ETA=1.15h
FOCAL local=02250/20000 global=52250 lr=9.822332e-05 loss=(0.06293,0.31737,0.09467) sec/step=0.230 ETA=1.13h
FOCAL local=02500/20000 global=5

## Один полный natural validation после 20k

Это единственное использование validation в этом experiment.


In [7]:
if METRICS_PATH.is_file() and RESULT_PATH.is_file():
    print("Evaluation already complete:", METRICS_PATH)

else:
    checkpoint = torch.load(
        FINAL, map_location="cpu", weights_only=False
    )
    assert checkpoint["assembly"] == ASSEMBLY
    assert int(checkpoint["local_step"]) == CONTINUATION_STEPS
    assert checkpoint["base_checkpoint_sha256"] == EXPECTED_BASE_SHA

    model = create_cnn_presence_intensity(
        channels=EXPECTED_CHANNELS
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()

    val_gen = ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="validation",
        config=profile_config,
        seed=42,
    )

    result = evaluate_presence_intensity_model(
        model,
        val_gen,
        device=device,
        batch_size=1,
        score_decimals=5,
        progress_every_batches=100,
        fail_on_cpu_fallback=True,
    )

    presence = result.metrics.loc[
        result.metrics["score"]
        == "presence_probability"
    ].copy()

    primary = presence[
        [
            "segment",
            "average_precision",
            "epic_spearman",
        ]
    ].copy()
    primary.to_csv(METRICS_PATH, index=False)

    intensity = result.metrics.loc[
        result.metrics["score"]
        == "intensity_prediction",
        ["segment", "epic_spearman"],
    ].rename(
        columns={
            "epic_spearman":
                "intensity_EPIC_Spearman_raw"
        }
    )

    strong = presence[
        [
            "segment",
            "positives",
            "average_precision",
            "epic_spearman",
            "quantized_spearman",
        ]
    ].rename(
        columns={
            "positives": "positive_n",
            "average_precision": "presence_AP_5dp",
            "epic_spearman":
                "presence_EPIC_Spearman_raw",
            "quantized_spearman":
                "presence_EPIC_Spearman_5dp",
        }
    ).merge(
        intensity,
        on="segment",
        validate="one_to_one",
    )
    strong.to_csv(
        STRONG_METRICS_PATH,
        index=False,
    )

    assert (
        result.processed_positions
        == EXPECTED_VAL_POSITIONS
    )
    assert (
        result.processed_positives
        == EXPECTED_VAL_POSITIVES
    )
    assert not result.fallback_warnings

    overall = primary.loc[
        primary["segment"] == "overall"
    ].iloc[0]

    focal_ap = float(overall["average_precision"])
    focal_sp = float(overall["epic_spearman"])

    summary = {
        "experiment_id": EXPERIMENT_ID,
        "assembly": ASSEMBLY,
        "base_step": BASE_STEP,
        "continuation_steps": CONTINUATION_STEPS,
        "global_step": (
            BASE_STEP + CONTINUATION_STEPS
        ),
        "focal_gamma": FOCAL_GAMMA,
        "focal_alpha": FOCAL_ALPHA,
        "checkpoint": str(FINAL),
        "checkpoint_sha256": sha256_file(FINAL),
        "validation_position_strand":
            int(result.processed_positions),
        "validation_positives":
            int(result.processed_positives),
        "focal_AP": focal_ap,
        "focal_EPIC_Spearman": focal_sp,
        "EXP045_AP": 0.158191438225949,
        "EXP047B_AP": 0.1700035818376157,
        "EXP045_EPIC_Spearman":
            0.15125935941947766,
        "EXP047B_EPIC_Spearman":
            0.1698961163371729,
        "delta_focal_vs_EXP045_AP":
            focal_ap - 0.158191438225949,
        "delta_focal_vs_EXP047B_AP":
            focal_ap - 0.1700035818376157,
        "delta_focal_vs_EXP045_Spearman":
            focal_sp - 0.15125935941947766,
        "delta_focal_vs_EXP047B_Spearman":
            focal_sp - 0.1698961163371729,
        "validation_used_for_training": False,
        "checkpoint_selected_on_validation": False,
        "caveat": (
            "Exploratory validation; split inspected "
            "in prior experiments."
        ),
    }

    RESULT_PATH.write_text(
        json.dumps(
            summary,
            ensure_ascii=False,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

    display(primary)
    print(
        json.dumps(
            summary,
            ensure_ascii=False,
            indent=2,
        )
    )

    del checkpoint, model, val_gen, result
    gc.collect()


tiles=     1/53710 positions=      634 positions/s=23,623
tiles=   100/53710 positions=   184526 positions/s=78,033
tiles=   200/53710 positions=   381312 positions/s=80,978
tiles=   300/53710 positions=   579586 positions/s=82,180
tiles=   400/53710 positions=   774616 positions/s=82,585
tiles=   500/53710 positions=   965744 positions/s=82,599
tiles=   600/53710 positions=  1160024 positions/s=82,646
tiles=   700/53710 positions=  1356748 positions/s=82,870
tiles=   800/53710 positions=  1545988 positions/s=82,499
tiles=   900/53710 positions=  1729542 positions/s=81,898
tiles=  1000/53710 positions=  1916654 positions/s=81,776
tiles=  1100/53710 positions=  2110016 positions/s=81,866
tiles=  1200/53710 positions=  2298690 positions/s=81,636
tiles=  1300/53710 positions=  2484916 positions/s=81,035
tiles=  1400/53710 positions=  2677334 positions/s=80,843
tiles=  1500/53710 positions=  2874022 positions/s=81,002
tiles=  1600/53710 positions=  3068758 positions/s=81,150
tiles=  1700/5

,segment,average_precision,epic_spearman
0,overall,0.156924,0.165764
1,NC_071299.1,0.156903,0.222244
2,NC_071301.1,0.171388,0.210880
3,NC_071314.1,0.196313,0.238143
4,NC_071316.1,0.100871,0.174609
5,NC_071317.1,0.153795,0.219047
6,NC_071320.1,0.134959,0.180214
7,NC_071321.1,0.144414,0.243948


{
  "experiment_id": "SUBMIT-MOTH-FOCAL-GAMMA2",
  "assembly": "ilPloInte3.2",
  "base_step": 50000,
  "continuation_steps": 20000,
  "global_step": 70000,
  "focal_gamma": 2.0,
  "focal_alpha": null,
  "checkpoint": "D:\\Projects\\EPIC\\EPIC\\artifacts\\experiments\\SUBMIT_ilPloInte3p2_FOCAL_GAMMA2_CONTINUATION_20K\\run_001\\focal_gamma2_final.pt",
  "checkpoint_sha256": "f6b49781c97338a4e6494a90ce87169d7140c72603b373bbe51cb2b31b27d8e2",
  "validation_position_strand": 104114200,
  "validation_positives": 162306,
  "focal_AP": 0.156924287988882,
  "focal_EPIC_Spearman": 0.1657640107190605,
  "EXP045_AP": 0.158191438225949,
  "EXP047B_AP": 0.1700035818376157,
  "EXP045_EPIC_Spearman": 0.15125935941947766,
  "EXP047B_EPIC_Spearman": 0.1698961163371729,
  "delta_focal_vs_EXP045_AP": -0.0012671502370669996,
  "delta_focal_vs_EXP047B_AP": -0.013079293848733714,
  "delta_focal_vs_EXP045_Spearman": 0.014504651299582838,
  "delta_focal_vs_EXP047B_Spearman": -0.004132105618112392,
  "validatio

In [8]:
# Compact final comparison.
summary = json.loads(
    RESULT_PATH.read_text(encoding="utf-8")
)

comparison = pd.DataFrame(
    [
        {
            "model": "EXP045 @50k",
            "AP": summary["EXP045_AP"],
            "EPIC_Spearman":
                summary["EXP045_EPIC_Spearman"],
        },
        {
            "model": "EXP047-B risk +20k",
            "AP": summary["EXP047B_AP"],
            "EPIC_Spearman":
                summary["EXP047B_EPIC_Spearman"],
        },
        {
            "model": "FOCAL gamma=2 +20k",
            "AP": summary["focal_AP"],
            "EPIC_Spearman":
                summary["focal_EPIC_Spearman"],
        },
    ]
)

display(comparison)

print(
    "Δ Focal vs EXP045 AP:",
    f"{summary['delta_focal_vs_EXP045_AP']:+.6f}",
)
print(
    "Δ Focal vs EXP047-B AP:",
    f"{summary['delta_focal_vs_EXP047B_AP']:+.6f}",
)
print(
    "Δ Focal vs EXP045 Spearman:",
    f"{summary['delta_focal_vs_EXP045_Spearman']:+.6f}",
)
print(
    "Δ Focal vs EXP047-B Spearman:",
    f"{summary['delta_focal_vs_EXP047B_Spearman']:+.6f}",
)
print("Result:", RESULT_PATH)


,model,AP,EPIC_Spearman
0,EXP045 @50k,0.158191,0.151259
1,EXP047-B risk +20k,0.170004,0.169896
2,FOCAL gamma=2 +20k,0.156924,0.165764


Δ Focal vs EXP045 AP: -0.001267
Δ Focal vs EXP047-B AP: -0.013079
Δ Focal vs EXP045 Spearman: +0.014505
Δ Focal vs EXP047-B Spearman: -0.004132
Result: D:\Projects\EPIC\EPIC\artifacts\experiments\SUBMIT_ilPloInte3p2_FOCAL_GAMMA2_CONTINUATION_20K\run_001\result_summary.json
